In [2]:
import sqlite3
import pandas as pd

DB_PATH = "../data/cookie_cats.db"
conn = sqlite3.connect(DB_PATH)

In [3]:
query = """
SELECT
    engagement_segment,
    SUM(CASE WHEN version = 'gate_30' THEN 1 ELSE 0 END) AS gate_30_users,
    SUM(CASE WHEN version = 'gate_40' THEN 1 ELSE 0 END) AS gate_40_users,
    COUNT(*) AS total_users
FROM analysis_view
WHERE is_outlier = 0
GROUP BY engagement_segment
ORDER BY total_users DESC
"""
pd.read_sql(query, conn)

,engagement_segment,gate_30_users,gate_40_users,total_users
0,light,15736,16259,31995
1,medium,15891,15497,31388
2,heavy,8531,9058,17589
3,power_user,2604,2618,5222
4,never_played,1937,2057,3994


In [4]:
query = """
SELECT
    engagement_segment,
    ROUND(AVG(CASE WHEN version = 'gate_30' THEN retention_7_int END) * 100, 2) AS gate_30_r7,
    ROUND(AVG(CASE WHEN version = 'gate_40' THEN retention_7_int END) * 100, 2) AS gate_40_r7,
    ROUND(
        (AVG(CASE WHEN version = 'gate_40' THEN retention_7_int END) -
         AVG(CASE WHEN version = 'gate_30' THEN retention_7_int END)) * 100,
        2
    ) AS diff_pp
FROM analysis_view
WHERE is_outlier = 0
GROUP BY engagement_segment
ORDER BY
    CASE engagement_segment
        WHEN 'never_played' THEN 1
        WHEN 'light' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'heavy' THEN 4
        WHEN 'power_user' THEN 5
    END
"""
pd.read_sql(query, conn)

,engagement_segment,gate_30_r7,gate_40_r7,diff_pp
0,never_played,0.83,0.63,-0.19
1,light,1.93,1.94,0.01
2,medium,12.03,10.81,-1.21
3,heavy,47.58,44.70,-2.88
4,power_user,84.91,85.03,0.12


In [5]:
query = """
SELECT
    engagement_segment,
    ROUND(AVG(CASE WHEN version = 'gate_30' THEN retention_1_int END) * 100, 2) AS gate_30_r1,
    ROUND(AVG(CASE WHEN version = 'gate_40' THEN retention_1_int END) * 100, 2) AS gate_40_r1,
    ROUND(
        (AVG(CASE WHEN version = 'gate_40' THEN retention_1_int END) -
         AVG(CASE WHEN version = 'gate_30' THEN retention_1_int END)) * 100,
        2
    ) AS diff_pp
FROM analysis_view
WHERE is_outlier = 0
GROUP BY engagement_segment
ORDER BY
    CASE engagement_segment
        WHEN 'never_played' THEN 1
        WHEN 'light' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'heavy' THEN 4
        WHEN 'power_user' THEN 5
    END
"""
pd.read_sql(query, conn)

,engagement_segment,gate_30_r1,gate_40_r1,diff_pp
0,never_played,2.12,2.24,0.12
1,light,12.48,12.49,0.01
2,medium,53.09,52.31,-0.78
3,heavy,83.84,82.76,-1.08
4,power_user,93.74,93.20,-0.54


## What I found

The overall result is that gate_30 wins by 0.82 pp on day-7 retention. But the segment breakdown shows the loss isn't spread evenly. It's concentrated in two segments:

| Segment | day-7 diff |
|---|---|
| never_played | -0.19 pp |
| light | +0.01 pp (tied) |
| medium | -1.21 pp |
| heavy | -2.88 pp |
| power_user | +0.12 pp |

Heavy players take the biggest hit by far. Power users actually do slightly better with gate_40 (though the gap is tiny and probably noise).

This makes sense if you think about it: light and never-played users don't engage enough to reach the gate, so it doesn't affect them. Power users push past anything. The players who really *feel* the gate are medium and heavy users — and for them, hitting the friction later (gate_40) hurts retention rather than helping.

The day-1 retention pattern is similar but the gaps are smaller. Medium and heavy players show the biggest day-1 drops too. So the gate_40 disadvantage shows up immediately and widens by day-7.

**Implication for the analysis going forward:**

The overall test result hides important variation. If I just reported "gate_40 lost by 0.8 pp," I'd be missing that some segments are tied and one segment is taking a 3 pp hit. The recommendation might be: **keep gate_30 overall, but the real insight is that mid-to-heavy engaged players are the most sensitive to gate placement.**

This also sets up the uplift modeling phase — can we predict in advance which players will be hurt by gate_40?